# AI Star Composer — مسار التعلم الآلي (تخرّج)

هذا النوتبوك يربط خطوات **تصدير الداتا → تدريب LSTM مشروط بالستايل والكوكب (`style_idx` + `planet_idx`) → توليد MIDI** بشكل مرتب للمذكرة.

**قبل التشغيل:** من الطرفية (على نفس بايثون الـ kernel إن أمكن):
```bash
pip install -r requirements.txt
pip install -r requirements-ml.txt
```
(الخلية الأولى تحاول تثبيت `midiutil` تلقائياً إن كان ناقصاً.)

**تشغيل Jupyter:** من جذر المشروع `AI_Star_Composer` (أو شغّل الخلية التالية؛ تبحث عن المجلد تلقائياً).

**Windows:** إذا ظهر خطأ `libiomp5md.dll` / OpenMP، الخلية الأولى تضبط `KMP_DUPLICATE_LIB_OK` (حل معتاد مع PyTorch + Conda).

In [59]:
import os
import subprocess
import sys
from pathlib import Path

# Windows + PyTorch في Jupyter/Conda: يمنع خطأ libiomp5md.dll (OpenMP مكرر)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "services" / "harmony_engine.py").exists() and (p / "ml").exists():
            return p
        p = p.parent
    raise RuntimeError(
        "لم يُعثر على جذر المشروع. افتح Jupyter من مجلد AI_Star_Composer أو cd إليه في أول خلية."
    )


ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# توليد MIDI يحتاج midiutil على نفس بايثون الـ kernel (Conda أحياناً بدونها)
try:
    import midiutil  # noqa: F401
except ImportError:
    print("تثبيت midiutil على هذا الـ kernel…")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "midiutil"],
        cwd=str(ROOT),
    )

print("جذر المشروع:", ROOT)

جذر المشروع: C:\Users\adler\OneDrive\Desktop\AI_Star_Composer


## 1) جلب NASA — كل الكواكب الثمانية (كاش محلي)

يحفظ `data/nasa_planets/<planet>.json` لكل كوكب. نافذة **120 يوماً** لزيادة النقاط. **2 ثانية** بين الطلبات احتراماً لـ API. يتطلب إنترنت.

In [61]:
import json
import shutil
import time

from scripts.bootstrap import init_environment
from scripts.data_fetcher import PLANET_IDS, fetch_velocity_dataset

# نفس ترتيب ALL_PLANETS في data_fetcher (لا تعتمد على استيراد الاسم — يفيد بعد Restart أو نسخ قديم)
ALL_PLANETS = tuple(PLANET_IDS.keys())

init_environment(".")

CACHE = ROOT / "data" / "nasa_planets"
CACHE.mkdir(parents=True, exist_ok=True)
MARS_JSON = ROOT / "data" / "mars_ml_pipeline.json"
MARS_JSON.parent.mkdir(parents=True, exist_ok=True)

for i, planet in enumerate(ALL_PLANETS):
    if i > 0:
        time.sleep(2.0)
    ds = fetch_velocity_dataset(planet_name=planet, days_count=120)
    path = CACHE / f"{planet.lower()}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"planet": ds["planet"], "points": ds["points"]}, f, ensure_ascii=False)
    print(planet, "->", path, "| نقاط:", len(ds["points"]))

# يبقى مسار مارس القديم متوافقاً مع بقية السكربتات
mars_path = CACHE / "mars.json"
if mars_path.is_file():
    shutil.copyfile(mars_path, MARS_JSON)
    print("نسخ كاش ->", MARS_JSON)

Mercury -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\mercury.json | نقاط: 121
Venus -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\venus.json | نقاط: 121
Earth -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\earth.json | نقاط: 121
Mars -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\mars.json | نقاط: 121
Jupiter -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\jupiter.json | نقاط: 121
Saturn -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\saturn.json | نقاط: 121
Uranus -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\uranus.json | نقاط: 121
Neptune -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\neptune.json | نقاط: 121
نسخ كاش -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\mars_ml_pipeline.json


## 2) تصدير تسلسلات موسومة (`style_idx` + `planet_idx`)

يجمع **كل الكواكب** من الكاش في `data/ml/style_sequences.jsonl` (4 ستايلات × أوضاع × 32 بذرة لكل كوكب؛ كل سطر يحمل كوكب المصدر).

**اختصار من الطرفية (نفس النتيجة تقريباً):** `python -m ml.bundle_local_planet_exports` → يكتب `data/ml/style_sequences_8planets.jsonl`.

In [62]:
import subprocess

from scripts.data_fetcher import PLANET_IDS

ALL_PLANETS = tuple(PLANET_IDS.keys())

STYLE_JSONL = ROOT / "data" / "ml" / "style_sequences.jsonl"
STYLE_JSONL.parent.mkdir(parents=True, exist_ok=True)
PLANET_CACHE = ROOT / "data" / "nasa_planets"
MARS_JSON = ROOT / "data" / "mars_ml_pipeline.json"

SEEDS = ",".join(str(i) for i in range(32))
if STYLE_JSONL.is_file():
    STYLE_JSONL.unlink()

pairs = []
for planet in ALL_PLANETS:
    pj = PLANET_CACHE / f"{planet.lower()}.json"
    if pj.is_file():
        pairs.append((planet, pj))
if not pairs and MARS_JSON.is_file():
    pairs.append(("Mars", MARS_JSON))
    print("تحذير: كاش الكواكب غير موجود — يُستخدم مارس فقط. شغّل الخلية السابقة لكل الكواكب.")

assert pairs, "لا يوجد JSON كواكب — شغّل خلية جلب NASA أو احفظ mars_ml_pipeline.json"

for idx, (planet, pj) in enumerate(pairs):
    cmd = [
        sys.executable,
        "-m",
        "ml.export_style_sequences",
        "--load-json",
        str(pj),
        "--output",
        str(STYLE_JSONL),
        "--modes",
        "baseline,ai",
        "--seeds",
        SEEDS,
    ]
    if idx > 0:
        cmd.append("--append")
    r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
    if r.stdout:
        print(r.stdout, end="")
    if r.stderr:
        print(r.stderr, end="")
    assert r.returncode == 0, f"فشل التصدير {planet}"
    print("OK", planet, pj)

Wrote 256 labeled style sequences -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\ml\style_sequences.jsonl
OK Mercury C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\mercury.json
Wrote 256 labeled style sequences -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\ml\style_sequences.jsonl
OK Venus C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\venus.json
Wrote 256 labeled style sequences -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\ml\style_sequences.jsonl
OK Earth C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\earth.json
Wrote 256 labeled style sequences -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\ml\style_sequences.jsonl
OK Mars C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_planets\mars.json
Wrote 256 labeled style sequences -> C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\ml\style_sequences.jsonl
OK Jupiter C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\data\nasa_

## 3) تدريب LSTM (ستايل + كوكب)

عند وجود `planet_idx` في الـ JSONL يظهر السجل `[style+planet]` ويُفعّل تضمين الكوكب تلقائياً.

يحفظ النموذج في `ml/checkpoints/note_lstm_style.pt` (ملف ثنائي — لا تفتحه كنص). يمكنك تغيير `--out` إلى مثلاً `note_lstm_style_planet.pt` لمطابقة `LSTM_CHECKPOINT_PATH` في `.env`.

In [63]:
import subprocess

# لازم torch على نفس بايثون الـ kernel (مو شرط الطرفية الثانية)
def _ensure_torch():
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__)"],
        cwd=str(ROOT),
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print("جارٍ تثبيت torch على هذا الـ kernel…")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "torch>=2.0", "numpy"],
            cwd=str(ROOT),
        )
    else:
        print("torch:", r.stdout.strip(), "| Python:", sys.executable)


_ensure_torch()

CKPT = ROOT / "ml" / "checkpoints" / "note_lstm_style.pt"
CKPT.parent.mkdir(parents=True, exist_ok=True)

EPOCHS = 35  # داتا أكبر → تدريب أطول يساعد غالباً
cmd = [
    sys.executable,
    "-m",
    "ml.train_sequence_lstm",
    "--data",
    str(STYLE_JSONL),
    "--out",
    str(CKPT),
    "--epochs",
    str(EPOCHS),
    "--batch",
    "32",
    "--seq-len",
    "32",
    "--device",
    "cpu",
]
r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
if r.stdout:
    print(r.stdout)
if r.stderr:
    print(r.stderr)
assert r.returncode == 0, "فشل التدريب — اقرأ STDERR أعلاه (غالباً torch قديم جداً أو ملف الداتا ناقص)"
print("Checkpoint:", CKPT)

torch: 2.5.1 | Python: c:\Users\adler\anaconda3\python.exe
epoch 1/35 [style]  loss=1.1932
epoch 2/35 [style]  loss=0.6087
epoch 3/35 [style]  loss=0.5048
epoch 4/35 [style]  loss=0.4574
epoch 5/35 [style]  loss=0.4282
epoch 6/35 [style]  loss=0.4090
epoch 7/35 [style]  loss=0.3917
epoch 8/35 [style]  loss=0.3788
epoch 9/35 [style]  loss=0.3665
epoch 10/35 [style]  loss=0.3592
epoch 11/35 [style]  loss=0.3486
epoch 12/35 [style]  loss=0.3415
epoch 13/35 [style]  loss=0.3330
epoch 14/35 [style]  loss=0.3271
epoch 15/35 [style]  loss=0.3203
epoch 16/35 [style]  loss=0.3142
epoch 17/35 [style]  loss=0.3109
epoch 18/35 [style]  loss=0.3051
epoch 19/35 [style]  loss=0.3001
epoch 20/35 [style]  loss=0.2950
epoch 21/35 [style]  loss=0.2894
epoch 22/35 [style]  loss=0.2866
epoch 23/35 [style]  loss=0.2804
epoch 24/35 [style]  loss=0.2773
epoch 25/35 [style]  loss=0.2727
epoch 26/35 [style]  loss=0.2680
epoch 27/35 [style]  loss=0.2646
epoch 28/35 [style]  loss=0.2599
epoch 29/35 [style]  loss=

## 4) توليد عيّنات MIDI

غيّر `--style` إلى: `calm` | `pop` | `study` | `cinematic`. مع checkpoint مدرّب بكوكب، غيّر `PLANET_FOR_LSTM` (أسماء: Mercury … Neptune) لسماع اختلاف الشرط.

In [65]:
import os

PLANET_FOR_LSTM = "Mars"  # Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune

out_dir = ROOT / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
_env = os.environ.copy()
_env.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

for st in ("pop", "study", "cinematic", "calm"):
    mid = out_dir / f"lstm_{st}.mid"
    r = subprocess.run(
        [
            sys.executable,
            "-m",
            "ml.generate_from_lstm",
            "--checkpoint",
            str(CKPT),
            "--style",
            st,
            "--planet",
            PLANET_FOR_LSTM,
            "--out",
            str(mid),
            "--steps",
            "96",
        ],
        cwd=str(ROOT),
        env=_env,
        capture_output=True,
        text=True,
    )
    if r.stdout:
        print(r.stdout, end="")
    if r.stderr:
        print(r.stderr, end="")
    assert r.returncode == 0, f"فشل توليد {st} — راجع الرسالة أعلاه"
    print("OK", mid)

Wrote C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_pop.mid (96 notes)  style=pop  temp=0.88 top_p=0.90
OK C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_pop.mid
Wrote C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_study.mid (96 notes)  style=study  temp=1.08 top_p=0.96
OK C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_study.mid
Wrote C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_cinematic.mid (96 notes)  style=cinematic  temp=0.94 top_p=0.92
OK C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_cinematic.mid
Wrote C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_calm.mid (96 notes)  style=calm  temp=1.02 top_p=0.94
OK C:\Users\adler\OneDrive\Desktop\AI_Star_Composer\outputs\lstm_calm.mid


## ملاحظات للمذكرة

- **مصدر الداتا:** NASA Horizons + طبقة السونيفيكيشن الرمزية (`generate_events`) لكل ستايل ولكل كوكب في التصدير.
- **النموذج:** LSTM مع `style_idx` (4 فئات) و`planet_idx` (8 كواكب) عند توفرهما في JSONL.
- **الملف الثنائي:** حسب `--out` في خلية التدريب (مثلاً `note_lstm_style.pt` أو `note_lstm_style_planet.pt`).
- **MAESTRO/Lakh:** مسار منفصل عبر `ml/ingest_external_midi.py` (انظر `ml/SOURCES_AND_LICENSES.txt`).